## Image Analysis with a Multimodal Model in Microsoft Foundry

The legacy **Azure Computer Vision Image Analysis API (v3.2 and v4.0)** is deprecated and
retires on **September 25, 2028**. Both `azure-cognitiveservices-vision-computervision`
and `azure-ai-vision-imageanalysis` fall under this retirement.

Following Microsoft's [migration guidance](https://learn.microsoft.com/en-us/azure/ai-services/computer-vision/migration-options),
this notebook uses a **multimodal GPT model (gpt-4o) deployed in Microsoft Foundry /
Azure OpenAI** to perform captioning, tagging, object and people detection, and text
reading (OCR) in a single call using structured JSON output.

> Note: A generative model returns descriptive results rather than the pixel-accurate
> bounding boxes the old SDK produced. If you need exact text coordinates for OCR,
> use Azure Document Intelligence (Read model) instead.


### Installing Dependencies and Libraries

In [ ]:
%pip install openai pydantic pillow python-dotenv requests

### Setting Up the Environment

In [5]:
from dotenv import load_dotenv
import os

load_dotenv()
aoai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
aoai_key = os.getenv("AZURE_OPENAI_KEY")
aoai_deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT")

### Creating our Azure OpenAI Client

In [8]:
from openai import AzureOpenAI

# api_version 2024-10-21 (or later) is required for structured outputs
client = AzureOpenAI(
    azure_endpoint=aoai_endpoint,
    api_key=aoai_key,
    api_version="2024-10-21",
)

### Define the Structured Output Schema

Instead of selecting `VisualFeatures`, we describe the fields we want the model to return.
The Azure OpenAI SDK enforces this schema so the model always replies with valid JSON.

In [9]:
from pydantic import BaseModel
from typing import List


class DetectedObject(BaseModel):
    name: str
    description: str


class ImageAnalysis(BaseModel):
    caption: str
    dense_captions: List[str]
    tags: List[str]
    objects: List[DetectedObject]
    people_count: int
    text: List[str]  # lines of any text (OCR) visible in the image


class OcrResult(BaseModel):
    lines: List[str]

### Get Image Analysis Results

In [13]:
images_to_analyze = [
    "https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/building.jpg",
    "https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/person.jpg",
    "https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/street.jpg"
]

for image_url in images_to_analyze:
    print(f"\nAnalyzing image: {image_url}\n")

    completion = client.chat.completions.parse(
        model=aoai_deployment,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an image analysis assistant. Analyze the image and return the "
                    "requested structured information. Use gender-neutral language in the caption. "
                    "For 'text', list every line of text visible in the image, or an empty list if none."
                ),
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Analyze this image."},
                    {"type": "image_url", "image_url": {"url": image_url}},
                ],
            },
        ],
        response_format=ImageAnalysis,
    )

    analysis = completion.choices[0].message.parsed

    print("Caption:")
    print(f"  {analysis.caption}")

    print("\nDense captions:")
    for caption in analysis.dense_captions:
        print(f"  {caption}")

    print("\nTags:")
    print("  " + ", ".join(analysis.tags))

    print("\nObjects:")
    for obj in analysis.objects:
        print(f"  {obj.name}: {obj.description}")

    print(f"\nPeople detected: {analysis.people_count}")

    print("\nText found in image:")
    if analysis.text:
        for line in analysis.text:
            print(f"  {line}")
    else:
        print("  (none)")


Analyzing image: https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/building.jpg

Caption:
  The image depicts a large classical-style building with a prominent dome, surrounded by green grass and trees, with a few individuals on the lawn.

Dense captions:
  A classical building with a dome and people walking on the lawn.
  The photograph shows a prominent white building under a partly cloudy sky.
  Several people are strolling on the green field in front of the structure.

Tags:
  architecture, dome, lawn, sky, trees, people

Objects:
  building: A large white building with a dome.
  people: Scattered individuals walking or standing on the grass field.
  trees: Green trees on both sides of the building.
  grass: Open grassy area in the foreground.

People detected: 3

Text found in image:
  (none)

Analyzing image: https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/person.jpg



### Use the OCR Capability to Read Handwritten Text

![handwritten.jpg](./Images/handwritten.png)

In [11]:
from PIL import Image
from io import BytesIO
import requests

ocr_image_url = "https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/handwritten.png"

# Download the image so we can display it alongside the transcription
response = requests.get(ocr_image_url)
original_image = Image.open(BytesIO(response.content)).convert("RGB")

# The multimodal model transcribes the handwriting; it does not return pixel coordinates
completion = client.beta.chat.completions.parse(
    model=aoai_deployment,
    messages=[
        {
            "role": "system",
            "content": (
                "You are an OCR assistant. Transcribe the text in the image exactly as written, "
                "returning one entry per visible line of text."
            ),
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Read the handwritten text in this image."},
                {"type": "image_url", "image_url": {"url": ocr_image_url}},
            ],
        },
    ],
    response_format=OcrResult,
)

ocr = completion.choices[0].message.parsed

print("Transcribed text:\n")
for line in ocr.lines:
    print(line)

original_image.show()

Transcribed text:

Hello,
Simply Noted has developed
incredible proprietary robotic
technology to write your message
and envelopes with a genuine
real pen. It is completely
indistinguishable from a humans
handwriting.
Try us today!
Simply Noted
